# 03 Portfolio Backtest

This notebook runs multiple symbols and combines them into an equal-weight portfolio.

Key outputs:

- Portfolio dashboard with combined equity, drawdown, per-symbol equity, and PnL contribution.
- Per-symbol contribution table to identify which symbols help or hurt the portfolio.
- Portfolio-level trade explorer.


In [ ]:
#
#

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Could not find repo root containing {marker!r} and core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:

from IPython.display import display

from core_python.strategies.combo.params import SYMBOLS, summary as strategy_summary
from core_python.strategies.combo.research_utils import (
    build_portfolio_replay_widget,
    configure_notebook,
    export_result_bundle,
    export_ctrader_validation_bundle,
    plot_portfolio_dashboard,
    show_ftmo_check,
    show_note,
    show_portfolio_summary,
    show_run_config,
    show_trade_explorer,
)
from core_python.strategies.combo.portfolio.backtest import run_portfolio_backtest

configure_notebook()
print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


In [ ]:
#

RUN_CONFIG = {
    'symbols': ['US30', 'US500', 'DE40', 'GOLD', 'BTCUSD'],
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'date_from': '2023-01-01',
    'date_to': None,
    'max_bars': 30_000,
    'indicator_overrides': {},
    'symbol_overrides': {},
    'collect_events': True,
    'portfolio_replay': {
        'selected_symbols': ['US30', 'US500', 'DE40', 'GOLD', 'BTCUSD'],
        'default_speed_ms': 300,
    },
    'export_report': False,
    'export_ctrader': False,
}

show_run_config('Portfolio Backtest Configuration', RUN_CONFIG)


In [ ]:
#
# Pipeline th?t theo code:
# 3. T?nh trade log, account equity, per-symbol equity v? metrics c?p portfolio.

portfolio = run_portfolio_backtest(
    symbol_keys=RUN_CONFIG['symbols'],
    initial_balance=RUN_CONFIG['initial_balance'],
    account_mode=RUN_CONFIG['account_mode'],
    date_from=RUN_CONFIG['date_from'],
    date_to=RUN_CONFIG['date_to'],
    indicator_overrides=RUN_CONFIG['indicator_overrides'] or None,
    symbol_overrides=RUN_CONFIG['symbol_overrides'] or None,
    max_bars=RUN_CONFIG['max_bars'],
    collect_events=RUN_CONFIG.get('collect_events', False),
)

print('Mode              =', portfolio.account_mode)
print('Symbols           =', portfolio.symbol_keys)
print('Portfolio trades  =', len(portfolio.trades))
print('Equity rows       =', len(portfolio.combined_equity))
if len(portfolio.combined_equity):
    print('Equity range      =', portfolio.combined_equity.index.min(), '->', portfolio.combined_equity.index.max())
    print('Final equity      =', round(float(portfolio.combined_equity.iloc[-1]), 2))


In [ ]:
#

per_symbol_df = show_portfolio_summary(portfolio)
show_ftmo_check(portfolio.metrics)


In [ ]:
#

plot_portfolio_dashboard(portfolio)


In [ ]:
#

portfolio_trades_df = show_trade_explorer(portfolio.trades, tail=60, title='Portfolio trade explorer')


In [ ]:
# Cell 8 - Portfolio replay dashboard
#
# This is a visual replay over the completed portfolio backtest. It does not
# place live orders and does not change the execution engine.

if 'portfolio' not in globals():
    print('No portfolio result yet. Run Cell 4 first.')
else:
    portfolio_replay = build_portfolio_replay_widget(portfolio, RUN_CONFIG)
    display(portfolio_replay)


In [ ]:

if RUN_CONFIG.get('export_report'):
    export_result_bundle(
        f"portfolio_{RUN_CONFIG['account_mode']}_backtest",
        metrics=portfolio.metrics,
        trades=portfolio.trades,
        equity=portfolio.combined_equity,
    )
else:
    print("Export is disabled. Set RUN_CONFIG['export_report'] = True to save CSV files.")


In [ ]:
# Cell 10 - Optional cTrader validation export

if 'portfolio' not in globals():
    print('No portfolio result yet. Run Cell 4 first.')
elif RUN_CONFIG.get('export_ctrader'):
    out = export_ctrader_validation_bundle(
        portfolio,
        RUN_CONFIG,
        name=f"portfolio_{RUN_CONFIG['account_mode']}_backtest",
    )
    print('cTrader validation export:', out)
else:
    print("cTrader export is disabled. Set RUN_CONFIG['export_ctrader'] = True to save validation CSV files.")
